In [1]:
from src.ingestion.client import create_igdb_client

client = create_igdb_client()
games = client.query("games", "fields name, rating; limit 3;")
print(games)


2026-04-11 23:15:41,378 | INFO | src.ingestion.auth | Requesting a new Twitch OAuth token for IGDB access.


[{'id': 350392, 'name': 'Rival Species'}, {'id': 207571, 'name': 'A Very Corporate Escape Room'}, {'id': 339266, 'name': 'Power Guy World'}]


In [24]:
import time
import datetime as dt
import requests
from src.utils.config import get_settings

settings = get_settings()
TOKEN_URL = "https://id.twitch.tv/oauth2/token"

# making an example object class for myself to practice
class tokenManager:
    def __init__(
            self,
            url=TOKEN_URL,
            client_id="",
            client_secret="",
            timeout=30,
    ):
        self.url = url
        self.client_id = client_id
        self.client_secret = client_secret
        self.timeout = timeout
        self.oauth_token = ""
        self.expires = 0
    
    def get_oauth_token(self):
        if not self.client_id or not self.client_secret:
            raise Exception("no client id or secret")
        
        response = requests.post(
            self.url,
            params={
                "client_id": self.client_id,
                "client_secret": self.client_secret,
                "grant_type": "client_credentials",
            },
            timeout=self.timeout,
        )
        response.raise_for_status()
        payload = response.json()

        self.oauth_token = payload.get("access_token", "")
        expires_in = payload.get("expires_in", 0)
        self.expires = time.time() + expires_in
        return self.oauth_token

    def is_token_expired(self):
        now = time.time()
        # giving ourselves 30 seconds of leeway
        return now >= self.expires - 30

    def return_oauth_token(self):
        if self.is_token_expired():
            print("Token expired, generating new token")
            return self.get_oauth_token()
        else:
            return self.oauth_token


In [25]:
# now testing to see if my oath functionality makes sense
client = tokenManager(client_id=settings.igdb_client_id, client_secret=settings.igdb_client_secret)
token = client.return_oauth_token()

print(token)
print(dt.datetime.fromtimestamp(client.expires))

Token expired, generating new token
r5c6jqyhzccae2lft36qb4r5bw913p
2026-06-21 10:48:13.924799


In [26]:
IGDB_BASE_URL = "https://api.igdb.com/v4"

class IGDBClient:

    def __init__(
      self,
      tokenClient: tokenManager,
      base_url=IGDB_BASE_URL,
      timeout=30,
    ):
        self.tokenClient = tokenClient
        self.base_url = base_url
        self.timeout = timeout

    def query(self, endpoint, query):
        url = self.base_url + f'/{endpoint}'

        access_token = self.tokenClient.return_oauth_token()
        headers = {
            "Accept": "application/json",
            "Client-ID": self.tokenClient.client_id,
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "text/plain",
        }

        response = requests.post(
            url,
            data=query,
            headers=headers,
            timeout=self.timeout,
        )
        response.raise_for_status()
        payload = response.json()

        return payload


In [33]:
game_client = IGDBClient(client)
data = game_client.query("games", "fields name, rating; limit 3;")
print(data)

[{'id': 350392, 'name': 'Rival Species'}, {'id': 207571, 'name': 'A Very Corporate Escape Room'}, {'id': 339266, 'name': 'Power Guy World'}]


In [ ]:
DEFAULT_GAME_FIELDS = (
    "id",
    "name",
    "slug",
    "first_release_date",
    "rating",
    "rating_count",
    "total_rating",
    "total_rating_count",
    "updated_at",
)

def get_game_data(game_client: IGDBClient, fields=list(DEFAULT_GAME_FIELDS), limit=5, batches=3):
    # loop through code for number of batches and query limit of data in each batch
    all_games = []
    offset = 0

    for i in range(batches):
        query = f"fields {', '.join(DEFAULT_GAME_FIELDS)}; limit {limit}; offset {offset};"
        data = game_client.query('games', query)
        all_games.extend(data)
        offset += limit

        # if returned fewer games than limit or nothing returned, then all games queried
        if not data or len(data) < limit:
            break
    
    return all_games

games = get_game_data(game_client)
print(len(games))

15


In [38]:
games

[{'id': 350392,
  'name': 'Rival Species',
  'slug': 'rival-species',
  'updated_at': 1756997209},
 {'id': 207571,
  'name': 'A Very Corporate Escape Room',
  'slug': 'a-very-corporate-escape-room',
  'updated_at': 1769233592},
 {'id': 339266,
  'first_release_date': 1692057600,
  'name': 'Power Guy World',
  'slug': 'power-guy-world',
  'updated_at': 1758827632},
 {'id': 72,
  'first_release_date': 1303084800,
  'name': 'Portal 2',
  'rating': 91.32833893724695,
  'rating_count': 4286,
  'slug': 'portal-2',
  'total_rating': 91.8863916908457,
  'total_rating_count': 4295,
  'updated_at': 1776708532},
 {'id': 63844,
  'first_release_date': 756518400,
  'name': 'Ace wo Nerae!',
  'rating': 52.90462943179914,
  'rating_count': 5,
  'slug': 'ace-wo-nerae',
  'total_rating': 52.90462943179914,
  'total_rating_count': 5,
  'updated_at': 1748464541},
 {'id': 371149,
  'name': 'The Deal',
  'slug': 'the-deal--2',
  'updated_at': 1759347838},
 {'id': 330684,
  'name': 'Nightmare Kart: The Old 